# 🔄 Recurrent Neural Network (RNN) — Practice Notebook

**This notebook contains guided exercises — implement the # TODO blocks.**

**Difficulty**: ⭐⭐ Intermediate  
**Time**: ~45 minutes

---


## 🎯 Section 1: Overview

A **Recurrent Neural Network (RNN)** is a class of artificial neural networks where connections between nodes form a directed graph along a temporal sequence. This allows it to exhibit temporal dynamic behavior, keeping a memory (hidden state) of previous sequence tokens.

### Applications
- Sentiment analysis
- Time-series forecasting
- Language translation (sequence-to-sequence)


## 📐 Section 2: Math & Intuition

### RNN Cell Forward Formulation
At time step $t$, given input $x_t$ and previous hidden state $h_{t-1}$:
$$a_t = x_t W_x + h_{t-1} W_h + b$$
$$h_t = \tanh(a_t)$$
where $h_0$ is typically initialized to zero.

### Backpropagation Through Time (BPTT)
RNNs backpropagate through time by unrolling the model across sequence length $T$. Gradients are accumulated across all time steps. However, multiplying by $W_h$ repeatedly at each step leads to gradients shrinking (vanishing) or growing exponentially (exploding).


## 🔧 Section 3: Implementation from Scratch


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
print('RNN Setup complete! ✅')


### 3.1 RNN Single-Step Forward Pass


In [ ]:
def rnn_cell_forward(xt, h_prev, Wx, Wh, b):
    """
    xt: input vector at time t of shape (batch_size, input_dim)
    h_prev: hidden state of previous step of shape (batch_size, hidden_dim)
    Wx: weight matrix for inputs of shape (input_dim, hidden_dim)
    Wh: weight matrix for hidden state of shape (hidden_dim, hidden_dim)
    b: bias vector of shape (1, hidden_dim)
    """
    # TODO: Implement the tanh update rule
    return h_next


### 3.2 RNN Full Sequence Forward Pass


In [ ]:
def rnn_forward(X, h0, Wx, Wh, b):
    """
    X: sequence matrix of shape (seq_len, batch_size, input_dim)
    h0: initial hidden state of shape (batch_size, hidden_dim)
    """
    seq_len, batch_size, input_dim = X.shape
    hidden_dim = h0.shape[1]
    
    h_states = np.zeros((seq_len, batch_size, hidden_dim))
    h_curr = h0
    
    # TODO: Loop over sequence length and perform recurrent updates
        
    return h_states


### 3.3 Verify Forward Passes


In [ ]:
X_dummy = np.random.randn(5, 2, 3)  # seq_len=5, batch_size=2, input_dim=3
h0_dummy = np.zeros((2, 4))         # hidden_dim=4
Wx_dummy = np.random.randn(3, 4)
Wh_dummy = np.random.randn(4, 4)
b_dummy = np.zeros((1, 4))

h_all = rnn_forward(X_dummy, h0_dummy, Wx_dummy, Wh_dummy, b_dummy)
print('RNN states output shape (should be [5, 2, 4]):', list(h_all.shape))
if 'TODO' not in rnn_cell_forward.__code__.co_consts and 'TODO' not in rnn_forward.__code__.co_consts:
    assert list(h_all.shape) == [5, 2, 4]
    print('RNN verification passed! ✅')


## 📦 Section 4: Library Implementation


We will define a sequence forecasting model using PyTorch's native `nn.RNN` module.


In [ ]:
import torch
import torch.nn as nn

class PyTorchRNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        # TODO: Define nn.RNN and classification layer
        
    def forward(self, x):
        # x shape: (batch, seq_len, input_dim)
        # TODO: Forward through rnn, take last hidden state, predict
        


In [ ]:
model = PyTorchRNN(input_dim=5, hidden_dim=10, output_dim=2)
dummy_seq = torch.randn(8, 15, 5)  # batch size of 8, sequence length of 15, input dimensions of 5
preds = model(dummy_seq)
print('Output shape (should be [8, 2]):', list(preds.shape))
assert list(preds.shape) == [8, 2]
print('PyTorch RNN compilation check passed! ✅')


## 🧪 Section 5: Experiments


Demonstrate the exploding gradient problem: when training an RNN on extremely long sequences, track the gradient norm.


In [ ]:
# Exploding gradient simulation
seq_len = 150
torch.manual_seed(0)
rnn = nn.RNN(input_size=1, hidden_size=1, num_layers=1, bias=False)
with torch.no_grad():
    # Set weight to a value > 1.0
    rnn.weight_hh_l0.fill_(1.5)
    rnn.weight_ih_l0.fill_(1.0)

x = torch.ones(seq_len, 1, 1, requires_grad=True)
h0 = torch.zeros(1, 1, 1)
out, hn = rnn(x, h0)

# Gradient of final hidden state w.r.t initial input
hn.backward()
grad_norm = x.grad.clone().squeeze()

plt.figure(figsize=(8, 4))
plt.plot(grad_norm.numpy(), color='red')
plt.yscale('log')
plt.title('Exploding Gradients in Vanilla RNN (Log Scale)')
plt.xlabel('Sequence Step')
plt.ylabel('Gradient Norm')
plt.show()
print(f'Gradient value at earliest sequence step: {grad_norm[0].item():.4e}')


## ❓ Section 6: Interview Questions


### Q1: Why do standard RNNs suffer from vanishing and exploding gradients?
**Answer**:
During Backpropagation Through Time (BPTT), the loss gradient at step $T$ is backpropagated to step $0$ by repeatedly multiplying by the recurrent weight matrix transpose $(W_{hh})^T$. If the largest eigenvalue of $W_{hh}$ is $> 1.0$, the gradient grows exponentially ($1.5^{150}$), causing exploding gradients. If the largest eigenvalue is $< 1.0$, the gradient decays exponentially to zero, preventing the weights from learning long-term dependencies.

### Q2: Explain Backpropagation Through Time (BPTT).
**Answer**:
BPTT is the standard backpropagation algorithm applied to sequential data. The RNN architecture is unrolled through all time steps of the sequence. The forward pass is computed for all steps, storing outputs and hidden states. In the backward pass, gradients are computed starting from the loss at the final time step and backpropagated backward through time, accumulating weight adjustments across all temporal steps.

### Q3: What is Teacher Forcing and when is it used?
**Answer**:
Teacher Forcing is a training method for recurrent networks where the model receives the ground-truth target sequence token as input at the next step, rather than feeding its own predicted output back into itself. This speeds up training and keeps the model stable early on, but can lead to 'exposure bias' during inference when the ground truth is unavailable.

### Q4: What is the difference between autoregressive and non-autoregressive decoding?
**Answer**:
- **Autoregressive decoding** generates tokens sequentially, where each output token relies on the previously generated tokens. Standard RNNs and GPT-style models use this.
- **Non-autoregressive decoding** attempts to predict all target outputs in parallel, which is faster but struggles to capture dependency correlations between output tokens.


## 🏆 Section 7: Challenge — BPTT Cell Gradient


**Challenge**: Derive and implement the parameter gradient updates for a single RNN cell backward step.


In [ ]:
def rnn_cell_backward(dnext, xt, h_prev, h_curr, Wx, Wh):
    """
    Compute gradients for a single step of the RNN cell.
    dnext: gradient of loss w.r.t current hidden state (batch_size, hidden_dim)
    xt: input at current step (batch_size, input_dim)
    h_prev: hidden state of previous step (batch_size, hidden_dim)
    h_curr: activated current hidden state (batch_size, hidden_dim)
    """
    # Gradient of tanh: dtanh = (1 - tanh²)
    # TODO: Implement gradient calculations w.r.t weights
    
    return dWx, dWh, db, dprev

# Check shapes
dnext_v = np.ones((1, 4))
xt_v = np.ones((1, 3))
h_prev_v = np.zeros((1, 4))
h_curr_v = np.ones((1, 4))
Wx_v = np.ones((3, 4))
Wh_v = np.ones((4, 4))

dWx, dWh, db, dprev = rnn_cell_backward(dnext_v, xt_v, h_prev_v, h_curr_v, Wx_v, Wh_v)
if 'TODO' not in rnn_cell_backward.__code__.co_consts:
    assert dWx.shape == (3, 4)
    assert dWh.shape == (4, 4)
    assert db.shape == (1, 4)
    assert dprev.shape == (1, 4)
    print('BPTT cell backward pass implementation verified! ✅')
